In [ ]:
import Pkg

Pkg.activate(".")

using Agents
using Distributions
using Random
using Statistics
using Printf
using ColorTypes
using StatsBase
using DataFrames
using Clustering
using RollingFunctions
using Graphs
using AgentsPlots
using CairoMakie
using Logging
using JLD2
using DataFrames
using Plots


include("agents.jl")
include("utils.jl")
include("model.jl")
include("Model_definitions_1.jl")

lista_seed_mista = [928154, 224468, 245633, 234277, 614952, 128866, 228554, 373344, 769051, 525478, 233024, 889337, 580, 215646, 9248]


  Activating project at `c:\Users\flavi\OneDrive\Desktop\materiale didattico\physics of complex systems\Tesi\Codici\Codici per articolo`

Cella 2: Struct agenti e funzioni helper definite.


plot_rent_heatmap (generic function with 1 method)

### Cella di simulazione per 15 seed e salvataggio dati

In [ ]:
# --- 1. Impostazioni dell'Esperimento ad Alta Risoluzione ---
# MODIFICA: Range di frazioni da 0.0 a 0.15 con passo 0.01
valori_frazione_da_testare = collect(0.0:0.01:0.15)
n_steps_dinamici_esperimento = 170

# MODIFICA: Aggiunta di un parametro per il boost del spillover negativo
# Valore inizializzato come positivo diventa negativo e viceversa
public_housing_negative_spillover_boost = 1.0


lista_seed_migliori = lista_seed_mista

println("Avvio esperimento ad alta risoluzione con $(length(lista_seed_migliori)) seed e $(length(valori_frazione_da_testare)) valori di frazione.")

# DataFrame per raccogliere TUTTI i risultati
master_model_results_df = DataFrame()
master_quadrant_results_df = DataFrame()

# --- 2. Ciclo dell'Esperimento ---
for frac in valori_frazione_da_testare
    println("\n=====================================================")
    println("--- INIZIO TEST CON public_housing_fraction = $(round(frac, digits=2)) ---")
    println("=====================================================")
    
    ph_fraction = frac

    # --- Esegui per lo scenario PH Random ---
    model_data_random, quadrant_data_random = run_multi_seed_scenario_new(
        lista_seed_migliori, 
        model_step_dinamica_public_random!,  
        n_steps_dyn=n_steps_dinamici_esperimento,
        public_housing_fraction=ph_fraction,
        public_housing_negative_spillover_boost=public_housing_negative_spillover_boost
    )
    model_data_random.policy_scenario .= "PH_Random"; model_data_random.ph_fraction .= frac
    quadrant_data_random.policy_scenario .= "PH_Random"; quadrant_data_random.ph_fraction .= frac
    append!(master_model_results_df, model_data_random)
    append!(master_quadrant_results_df, quadrant_data_random)

    # --- Esegui per lo scenario PH Rent-Gap ---
    model_data_rentgap, quadrant_data_rentgap = run_multi_seed_scenario_new(
        lista_seed_migliori, 
        model_step_dinamica_public_rent_gap!, 
        n_steps_dyn=n_steps_dinamici_esperimento,
        public_housing_fraction=ph_fraction,
        public_housing_negative_spillover_boost=public_housing_negative_spillover_boost

    )
    model_data_rentgap.policy_scenario .= "PH_RentGap"; model_data_rentgap.ph_fraction .= frac
    quadrant_data_rentgap.policy_scenario .= "PH_RentGap"; quadrant_data_rentgap.ph_fraction .= frac
    append!(master_model_results_df, model_data_rentgap)
    append!(master_quadrant_results_df, quadrant_data_rentgap)
end

println("\n✅ Esperimento di sensitività ad alta risoluzione completato.")

# --- 3. Salva i risultati finali! ---
# MODIFICA: Nuovo nome per il file dei risultati
nome_file_output = "nome_risultati.jld2"
JLD2.save(nome_file_output, Dict(
    "master_model_results" => master_model_results_df,
    "master_quadrant_results" => master_quadrant_results_df,
    "valori_frazione_usati" => valori_frazione_da_testare,
    "lista_seed_usati" => lista_seed_migliori
))
println("Dati dell'esperimento ad alta risoluzione salvati con successo nel file: $(nome_file_output)")
    


Avvio esperimento ad alta risoluzione con 1 seed e 3 valori di frazione.

--- INIZIO TEST CON public_housing_fraction = 0.0 ---
--- Avvio esecuzione per: model_step_dinamica_public_random! ---
  -> Seed: 580...

┌ Warning: From version 6.0 it is necessary to pass at least one of agent_step! or model_step!
│ as keywords argument when defining the model. The old version is deprecated.
│ Passing these functions to methods of the library which required them before
│ version 6.0 is also deprecated since they can be retrieved from the model instance.
│ In particular this means it is not needed to pass the stepping functions in step!,
│ run!, offline_run!, ensemblerun!, abmplot, abmplot!, abmexploration, abmvideo and
│ ABMObservable.
└ @ Agents C:\Users\flavi\.julia\packages\Agents\WuWeG\src\core\model_standard.jl:145
┌ Warning: Agent type is not concrete. If your agent is parametrically typed, you're probably
│ seeing this warning because you gave `Agent` instead of `Agent{Float64}`
│ (for example) to this function. You can also create an instance of your agent
│ and pass it to this function. If you want to use `Union` types for mixed agent
│ models, you can silence this warning.
│ If you are using 

--- Inizio Term (Steps da 1 a 15) ---


┌ Warning: Passing agent_step! and model_step! to step! is deprecated. Use the new version
│              step!(model, n = 1, agents_first = true)
└ @ Agents C:\Users\flavi\.julia\packages\Agents\WuWeG\src\deprecations.jl:97


Completato step di Term: 10 / 15 (Step assoluto: 10)
Completato step di Term: 15 / 15 (Step assoluto: 15)
--- Fine Term ---
--- Inizio Dyn (Steps da 16 a 25) ---
Completato step di Dyn: 10 / 10 (Step assoluto: 25)
--- Fine Dyn ---
 Fatto.
--- Avvio esecuzione per: model_step_dinamica_public_rent_gap! ---
  -> Seed: 580...--- Inizio Term (Steps da 1 a 15) ---
Completato step di Term: 10 / 15 (Step assoluto: 10)
Completato step di Term: 15 / 15 (Step assoluto: 15)
--- Fine Term ---
--- Inizio Dyn (Steps da 16 a 25) ---
Completato step di Dyn: 10 / 10 (Step assoluto: 25)
--- Fine Dyn ---
 Fatto.

--- INIZIO TEST CON public_housing_fraction = 0.05 ---
--- Avvio esecuzione per: model_step_dinamica_public_random! ---
  -> Seed: 580...--- Inizio Term (Steps da 1 a 15) ---
Completato step di Term: 10 / 15 (Step assoluto: 10)
Completato step di Term: 15 / 15 (Step assoluto: 15)
--- Fine Term ---
--- Inizio Dyn (Steps da 16 a 25) ---
Completato step di Dyn: 10 / 10 (Step assoluto: 25)
--- Fine D

### Cella di plot dei dati raccolti

In [ ]:

println("--- AVVIO SCRIPT DI ANALISI COMPLETA ---")

# =======================================================================
#               PASSO 1: CARICAMENTO E PREPARAZIONE
# =======================================================================

# --- Definisci qui il file che vuoi analizzare ---
nome_file_esperimento = "nome_risultati.jld2" 

println("--- Caricamento Dati da: $(nome_file_esperimento) ---")
dati_caricati = JLD2.load(nome_file_esperimento)
master_model_results_df = dati_caricati["master_model_results"]
valori_da_plottare = sort(unique(master_model_results_df.ph_fraction))
println("Dati caricati. Frazioni da analizzare: $(valori_da_plottare)")

# =======================================================================
#               PASSO 2: AGGREGAZIONE DATI
# =======================================================================

println("--- Aggregazione dei dati in corso... ---")
sem_robust(x) = begin; clean_x = collect(skipmissing(x)); n = length(clean_x); return n > 1 ? std(clean_x) / sqrt(n) : 0.0; end

# --- Aggregazione per i grafici di SINTESI FINALE ---
final_step = maximum(master_model_results_df.step)
final_step_data = filter(row -> row.step == final_step, master_model_results_df)
summary_final = combine(groupby(final_step_data, [:ph_fraction, :policy_scenario]),
    :median_inner_city_rent => mean ∘ skipmissing => :mean_final_rent, :median_inner_city_rent => sem_robust  => :sem_final_rent,
    :median_income_inner_city => mean ∘ skipmissing => :mean_final_income, :median_income_inner_city => sem_robust  => :sem_final_income,
    #:neighbour_segregation_index => mean ∘ skipmissing => :mean_final_neighbour_seg, :neighbour_segregation_index => sem_robust  => :sem_final_neighbour_seg
)

# --- Aggregazione per i grafici dell'EVOLUZIONE DINAMICA ---
summary_by_step = combine(groupby(master_model_results_df, [:policy_scenario, :ph_fraction, :step]),
    :median_inner_city_rent => mean ∘ skipmissing => :mean_rent, :median_inner_city_rent => sem_robust => :sem_rent,
    :median_income_inner_city => mean ∘ skipmissing => :mean_income, :median_income_inner_city => sem_robust => :sem_income,
    #:neighbour_segregation_index => mean ∘ skipmissing => :mean_seg_neighbour, :neighbour_segregation_index => sem_robust => :sem_seg_neighbour,
    :n_low_inner => mean ∘ skipmissing => :mean_n_low_inner, :n_low_inner => sem_robust => :sem_n_low_inner,
    :n_high_inner => mean ∘ skipmissing => :mean_n_high_inner, :n_high_inner => sem_robust => :sem_n_high_inner
)
println("Tutte le aggregazioni completate.")


# =======================================================================
#               PASSO 3: PLOTTING
# =======================================================================

# --- SEZIONE 3A: GRAFICI DI SINTESI FINALE (Valore Finale vs Frazione) ---
println("\n--- Generazione Grafici di Sintesi Finale ---")
p_rent_final = Plots.plot(title="Final Median Rent vs Public Housing Fraction", xlabel="Public Housing Fraction", ylabel="Final Median Rent", legend=:best, legendfontsize=9)
p_income_final = Plots.plot(#=title="Final Median Income vs Public Housing Fraction",=# xlabel="Public Housing Fraction", ylabel="Final Median Income", legend=:best,legendfontsize=9)
#p_seg_final = Plots.plot(title="Segregazione Finale (Neighbour) vs Frazione PH", xlabel="Frazione Public Housing", ylabel="Indice di Segregazione (Media ± SEM)", legend=:best)

for policy in ["PH_Random", "PH_RentGap"]
    policy_data = filter(row -> row.policy_scenario == policy, summary_final)
    sort!(policy_data, :ph_fraction)
    Plots.plot!(p_rent_final, policy_data.ph_fraction, policy_data.mean_final_rent, yerror=policy_data.sem_final_rent, label=policy, lw=2.5, marker=:circle)
    Plots.plot!(p_income_final, policy_data.ph_fraction, policy_data.mean_final_income, yerror=policy_data.sem_final_income, label=policy, lw=2.5, marker=:circle)
    #Plots.plot!(p_seg_final, policy_data.ph_fraction, policy_data.mean_final_neighbour_seg, yerror=policy_data.sem_final_neighbour_seg, label=policy, lw=2.5, marker=:circle)
end
display(p_rent_final); display(p_income_final); #display(p_seg_final)
savefig(p_income_final, "final median income.pdf")

# --- SEZIONE 3B: GRAFICI DELL'EVOLUZIONE DINAMICA ("a Spaghetti") ---
println("\n--- Generazione Grafici Dinamici ---")
n_steps_term = 15
n_frac = length(valori_da_plottare)
blue_palette = cgrad(:blues, n_frac, rev=true)
red_palette = cgrad(:reds, n_frac, rev=true)

for policy_scenario_name in ["PH_Random", "PH_RentGap"]
    println("\n--- Grafici per lo scenario: $(policy_scenario_name) ---")
    dati_plot = filter(row -> row.policy_scenario == policy_scenario_name, summary_by_step)
    
    # Crea i 4 plot vuoti per questo scenario
    p_rent = Plots.plot(title="Affitti ($(policy_scenario_name)) per Frazione", xlabel="Step", ylabel="Affitto Mediano", legend=:best)
    p_income = Plots.plot(title="Redditi ($(policy_scenario_name)) per Frazione", xlabel="Step", ylabel="Reddito Mediano", legend=:best)
    #p_seg = Plots.plot(title="Segregazione 'Neighbour' ($(policy_scenario_name))", xlabel="Step", ylabel="Indice Segregazione", legend=:best)
    p_agents = Plots.plot(title="Numero Agenti Inner City ($(policy_scenario_name))", xlabel="Step", ylabel="Numero Agenti", legend=:best)

    for (i, frac) in enumerate(valori_da_plottare)
        data_frac = filter(row -> row.ph_fraction == frac, dati_plot)
        if isempty(data_frac); continue; end
        Plots.plot!(p_rent, data_frac.step, data_frac.mean_rent, ribbon=data_frac.sem_rent, label="Frazione $frac", lw=2, fillalpha=0.15)
        Plots.plot!(p_income, data_frac.step, data_frac.mean_income, ribbon=data_frac.sem_income, label="Frazione $frac", lw=2, fillalpha=0.15)
        #Plots.plot!(p_seg, data_frac.step, data_frac.mean_seg_neighbour, ribbon=data_frac.sem_seg_neighbour, label="Frazione $frac", lw=2, fillalpha=0.15)
        Plots.plot!(p_agents, data_frac.step, data_frac.mean_n_low_inner, ribbon=data_frac.sem_n_low_inner, label="Low Inc (Frac $frac)", lw=2, linecolor=blue_palette[i], fillalpha=0.1)
        Plots.plot!(p_agents, data_frac.step, data_frac.mean_n_high_inner, ribbon=data_frac.sem_n_high_inner, label="High Inc (Frac $frac)", lw=2, linecolor=red_palette[i], fillalpha=0.1)
    end
    
    Plots.vline!(p_rent, [n_steps_term], label="", color=:black, linestyle=:dash); display(p_rent)
    Plots.vline!(p_income, [n_steps_term], label="", color=:black, linestyle=:dash); display(p_income)
    #Plots.vline!(p_seg, [n_steps_term], label="", color=:black, linestyle=:dash); display(p_seg)
    Plots.vline!(p_agents, [n_steps_term], label="", color=:black, linestyle=:dash); display(p_agents)
end

println("\n✅ Analisi e plotting completati.")